In [3]:
import cv2
import numpy as np
import time
import tempfile
from pathlib import Path
from IPython.display import display, Markdown

try:
    import google.colab  # noqa: F401
    from google.colab.patches import cv2_imshow
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    import matplotlib.pyplot as plt

if not IN_COLAB:
    get_ipython().run_line_magic("pip", "install -q opencv-python numpy ipywidgets matplotlib")


def show_frame(bgr):
    if IN_COLAB:
        cv2_imshow(bgr)
    else:
        plt.figure(figsize=(12, 3))
        plt.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
        plt.axis("off")
        plt.tight_layout()
        plt.show()


print("로컬 커널" if not IN_COLAB else "Colab 커널 (Cursor에서는 업로드 불가 → 로컬 커널 권장)")

def calculate_blockiness_heatmap(gray_frame):
    """
    프레임 내의 8x8 픽셀 격자 경계에서 발생하는 압축 손실(단차)을 계산하여
    시각적인 히트맵(Heatmap) 형태와 전체/지역별 손실 점수를 반환합니다.
    """
    h, w = gray_frame.shape
    mask = np.zeros((h, w), dtype=np.float32)
    
    # 1. H.264/JPEG의 기본 압축 단위인 8x8 격자의 경계면 픽셀 차이 계산
    gray_float = gray_frame.astype(np.float32)
    
    # 정확한 8x8 경계선 인덱스 추출
    h_bnd = np.arange(7, h - 1, 8)
    w_bnd = np.arange(7, w - 1, 8)
    
    # 수평/수직 경계선에서의 픽셀 단차(차이) 계산
    diff_h = np.abs(gray_float[h_bnd, :] - gray_float[h_bnd + 1, :])
    diff_v = np.abs(gray_float[:, w_bnd] - gray_float[:, w_bnd + 1])
    
    # 마스크에 단차 값 매핑
    mask[h_bnd, :] += diff_h
    mask[h_bnd + 1, :] += diff_h
    mask[:, w_bnd] += diff_v
    mask[:, w_bnd + 1] += diff_v
    
    # 2. 전체 프레임 평균 점수 계산
    raw_score = (np.mean(diff_h) + np.mean(diff_v)) / 2.0
    current_score = min(raw_score * 10.0, 100.0)
    
    # 3. 지역별(3x3 그리드) 분석
    grid_h, grid_w = h // 3, w // 3
    max_grid_score = 0
    max_grid_loc = ""
    
    locations = [
        ("Top-Left", 0, grid_h, 0, grid_w), ("Top-Center", 0, grid_h, grid_w, grid_w*2), ("Top-Right", 0, grid_h, grid_w*2, w),
        ("Mid-Left", grid_h, grid_h*2, 0, grid_w), ("Center", grid_h, grid_h*2, grid_w, grid_w*2), ("Mid-Right", grid_h, grid_h*2, grid_w*2, w),
        ("Bot-Left", grid_h*2, h, 0, grid_w), ("Bot-Center", grid_h*2, h, grid_w, grid_w*2), ("Bot-Right", grid_h*2, h, grid_w*2, w)
    ]
    
    for name, y1, y2, x1, x2 in locations:
        grid_mask = mask[y1:y2, x1:x2]
        non_zero_vals = grid_mask[grid_mask > 0]
        if len(non_zero_vals) > 0:
            grid_score = min(np.mean(non_zero_vals) * 5.0, 100.0) # 스케일링 보정
            if grid_score > max_grid_score:
                max_grid_score = grid_score
                max_grid_loc = name

    # 4. 히트맵 생성
    heatmap_blurred = cv2.GaussianBlur(mask, (15, 15), 0)
    heatmap_absolute = np.clip(heatmap_blurred * 10.0, 0, 255).astype(np.uint8)
    color_map = cv2.applyColorMap(heatmap_absolute, cv2.COLORMAP_JET)
    
    return color_map, current_score, max_grid_loc, max_grid_score

def calculate_blur_score(gray_frame):
    return cv2.Laplacian(gray_frame, cv2.CV_64F).var()

def calculate_fft_grid_peak(gray_frame):
    f_transform = np.fft.fft2(gray_frame)
    f_shift = np.fft.fftshift(f_transform)
    magnitude_spectrum = 20 * np.log(np.abs(f_shift) + 1)
    
    h, w = magnitude_spectrum.shape
    cy, cx = h // 2, w // 2
    
    mask_radius = 30
    Y, X = np.ogrid[:h, :w]
    dist_from_center = np.sqrt((X - cx)**2 + (Y - cy)**2)
    high_freq_region = magnitude_spectrum[dist_from_center > mask_radius]
    
    peak_strength = np.std(high_freq_region) / 100.0 if len(high_freq_region) > 0 else 0
    return min(float(peak_strength), 1.0)

def run_blockiness_viewer(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"'{video_path}' 영상을 열 수 없습니다.")
        return

    print("프레임별 종합 화질/압축 컨텍스트 분석을 시작합니다...")
    frame_idx = 0
    metrics_timeline = {"blur": [], "blockiness": [], "fft_peak": []}

    while True:
        ret, frame = cap.read()
        if not ret: break
        frame_idx += 1
        
        if frame_idx % 10 != 0: continue
            
        gray_original = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        # 그리드 분석 결과
        heatmap_original, blockiness_score, worst_loc, worst_score = calculate_blockiness_heatmap(gray_original)
        blur_score = calculate_blur_score(gray_original)
        fft_peak_score = calculate_fft_grid_peak(gray_original)
        
        metrics_timeline["blockiness"].append(blockiness_score)
        metrics_timeline["blur"].append(blur_score)
        metrics_timeline["fft_peak"].append(fft_peak_score)
        
        target_w, target_h = 480, 270
        frame_resized = cv2.resize(frame, (target_w, target_h))
        heatmap_resized = cv2.resize(heatmap_original, (target_w, target_h))
        overlay = cv2.addWeighted(frame_resized, 0.4, heatmap_resized, 0.6, 0)
        
        # ---------------------------------------------------------
        # [신규 기능] 히트맵 위에 3x3 격자 그리기 및 시각적 하이라이트
        # ---------------------------------------------------------
        grid_w, grid_h = target_w // 3, target_h // 3
        
        # 1. 얇은 흰색 반투명 선으로 3x3 전체 격자 그리기
        for i in range(1, 3):
            # 수직선
            cv2.line(overlay, (i * grid_w, 0), (i * grid_w, target_h), (255, 255, 255), 1, cv2.LINE_AA)
            # 수평선
            cv2.line(overlay, (0, i * grid_h), (target_w, i * grid_h), (255, 255, 255), 1, cv2.LINE_AA)

        # 구역별 좌표 매핑 (x1, y1, x2, y2)
        grid_coords = {
            "Top-Left": (0, 0, grid_w, grid_h), "Top-Center": (grid_w, 0, grid_w*2, grid_h), "Top-Right": (grid_w*2, 0, target_w, grid_h),
            "Mid-Left": (0, grid_h, grid_w, grid_h*2), "Center": (grid_w, grid_h, grid_w*2, grid_h*2), "Mid-Right": (grid_w*2, grid_h, target_w, grid_h*2),
            "Bot-Left": (0, grid_h*2, grid_w, target_h), "Bot-Center": (grid_w, grid_h*2, grid_w*2, target_h), "Bot-Right": (grid_w*2, grid_h*2, target_w, target_h)
        }

        # 2. 가장 심하게 훼손된 구역(Worst Area) 붉은색 테두리로 강조
        if worst_score > 20 and worst_loc in grid_coords:
            x1, y1, x2, y2 = grid_coords[worst_loc]
            # 굵은 붉은색 테두리
            cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 0, 255), 3)
            # 해당 칸 중앙에 점수 직접 표시
            text_x = x1 + (grid_w // 2) - 20
            text_y = y1 + (grid_h // 2) + 5
            cv2.putText(overlay, f"{worst_score:.1f}", (text_x, text_y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
        # ---------------------------------------------------------
        
        # --- UI 텍스트 추가 (왼쪽 원본 영상) ---
        cv2.putText(frame_resized, f"Frame: {frame_idx}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        cv2.putText(frame_resized, "[Metrics]", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        
        blur_color = (0, 0, 255) if blur_score < 100 else (0, 255, 0)
        cv2.putText(frame_resized, f"Blur: {blur_score:.1f}", (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.6, blur_color, 2)
        
        fft_color = (0, 0, 255) if fft_peak_score > 0.4 else (0, 255, 255)
        cv2.putText(frame_resized, f"FFT Peak: {fft_peak_score:.2f}", (10, 120), cv2.FONT_HERSHEY_SIMPLEX, 0.6, fft_color, 2)
        
        # --- UI 텍스트 추가 (오른쪽 히트맵 영상) ---
        status_color = (0, 0, 255) if blockiness_score > 30 else (0, 255, 255)
        cv2.putText(overlay, f"Avg Loss: {blockiness_score:.1f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, status_color, 2)
        
        if worst_score <= 20:
            cv2.putText(overlay, "Area Loss: Uniform/Clean", (10, target_h - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        
        combined_view = np.hstack((frame_resized, overlay))
        print(f"\n--- Frame {frame_idx} ---")
        show_frame(combined_view)
        time.sleep(0.1)

    cap.release()

    n_samples = len(metrics_timeline["blur"])
    lines = [
        "### 종합 분석 요약",
        f"- 총 **{frame_idx}** 프레임 중 **{n_samples}**개 샘플링",
    ]
    if n_samples:
        lines += [
            f"- **Blur** 평균: {np.mean(metrics_timeline['blur']):.1f} | 최저: {np.min(metrics_timeline['blur']):.1f}",
            f"- **Blockiness** 평균: {np.mean(metrics_timeline['blockiness']):.1f} | 최고: {np.max(metrics_timeline['blockiness']):.1f}",
            f"- **FFT Peak** 평균: {np.mean(metrics_timeline['fft_peak']):.2f} | 최고: {np.max(metrics_timeline['fft_peak']):.2f}",
        ]
    else:
        lines.append("- 샘플링된 프레임 없음 (10프레임마다 추출 — 영상이 짧으면 통계 없음)")

    summary_md = "\n".join(lines)
    display(Markdown(summary_md))
    return summary_md


Note: you may need to restart the kernel to use updated packages.
로컬 커널


In [4]:
# ▼ Cursor: Select Kernel → 로컬 Python (Anaconda base 등)
import ipywidgets as widgets
from IPython.display import display

if IN_COLAB:
    raise RuntimeError(
        "Cursor + Colab 커널에서는 파일 업로드가 안 됩니다.\n"
        "우측 상단 Select Kernel → Python 3.x (로컬) 로 바꾼 뒤 셀 0→1 순서로 다시 실행하세요."
    )

uploader = widgets.FileUpload(
    accept=".mp4,.avi,.mov,.mkv,.webm",
    multiple=False,
    description="영상 선택",
    button_style="primary",
)
run_btn = widgets.Button(description="분석 시작", button_style="success")
status = widgets.HTML("① 영상 선택 → ② 분석 시작")
output_area = widgets.Output()


def _parse_upload(val):
    if not val:
        return None, None
    if isinstance(val, (tuple, list)):
        info = val[0]
        return info["name"], bytes(info["content"])
    name = next(iter(val))
    item = val[name]
    content = item["content"] if isinstance(item, dict) else item
    return name, bytes(content)


def _on_run(_):
    name, content = _parse_upload(uploader.value)
    if not name:
        status.value = '<b style="color:red">영상을 먼저 선택하세요.</b>'
        return
    path = Path(tempfile.gettempdir()) / Path(name).name
    path.write_bytes(content)
    status.value = f"<b>분석 중:</b> {name}"
    with output_area:
        output_area.clear_output(wait=True)
        summary = run_blockiness_viewer(str(path))
        if summary:
            status.value = f"<b>완료:</b> {name}"


run_btn.on_click(_on_run)
display(widgets.VBox([uploader, run_btn, status, output_area]))
